In [37]:
!pip install nltk==3.8.1

In [2]:
#################### Data Processing ######################
from numpy import array
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.utils import to_categorical

##################### Model building #####################
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers import Embedding

In [3]:
data="California is a state in the Western United States. California borders Oregon to the north, Nevada and Arizona to the east, the Mexican state of Baja California to the south; and has a coastline along the Pacific Ocean to the west."

data

'California is a state in the Western United States. California borders Oregon to the north, Nevada and Arizona to the east, the Mexican state of Baja California to the south; and has a coastline along the Pacific Ocean to the west.'

# Data Pre-Processing 

#### Lower Case

In [4]:
# cleaning the data
data= data.lower()           # Converting the string to lower case to get uniformity
data

'california is a state in the western united states. california borders oregon to the north, nevada and arizona to the east, the mexican state of baja california to the south; and has a coastline along the pacific ocean to the west.'

#### Punctuation Case

In [5]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [6]:
def remove_punctuation(text):
    text_nopunt="".join([c 
                         for c in text
                         if c not in string.punctuation])
    return text_nopunt

In [7]:
data

'california is a state in the western united states. california borders oregon to the north, nevada and arizona to the east, the mexican state of baja california to the south; and has a coastline along the pacific ocean to the west.'

In [8]:
data = remove_punctuation(data)

data

'california is a state in the western united states california borders oregon to the north nevada and arizona to the east the mexican state of baja california to the south and has a coastline along the pacific ocean to the west'

#### Stopwords

In [9]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [10]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))
  
text_tokens = word_tokenize(data)
print("Word Token:  \n",text_tokens)

Word Token:  
 ['california', 'is', 'a', 'state', 'in', 'the', 'western', 'united', 'states', 'california', 'borders', 'oregon', 'to', 'the', 'north', 'nevada', 'and', 'arizona', 'to', 'the', 'east', 'the', 'mexican', 'state', 'of', 'baja', 'california', 'to', 'the', 'south', 'and', 'has', 'a', 'coastline', 'along', 'the', 'pacific', 'ocean', 'to', 'the', 'west']


In [11]:
tokens_without_sw = [word 
                     for word in text_tokens 
                     if not word in stopwords.words()]
print("Word Without StopWords:  \n",tokens_without_sw)

Word Without StopWords:  
 ['california', 'state', 'western', 'united', 'states', 'california', 'borders', 'oregon', 'north', 'nevada', 'arizona', 'east', 'mexican', 'state', 'baja', 'california', 'south', 'coastline', 'pacific', 'ocean', 'west']


In [12]:
data = (" ").join(tokens_without_sw)
data

'california state western united states california borders oregon north nevada arizona east mexican state baja california south coastline pacific ocean west'

#### Sequence Order

In [13]:
# Instantiating the Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])  ## convert sentance to word  
sequence_data = tokenizer.texts_to_sequences([data])[0]  # mode concept for sequence 
sequence_data  

[1, 2, 3, 4, 5, 1, 6, 7, 8, 9, 10, 11, 12, 2, 13, 1, 14, 15, 16, 17, 18]

In [14]:
# Getting the total number of words of the data.
word2idx = tokenizer.word_index  ####### index number to every token ro word 
print(len(word2idx))

18


In [15]:
print(word2idx)

{'california': 1, 'state': 2, 'western': 3, 'united': 4, 'states': 5, 'borders': 6, 'oregon': 7, 'north': 8, 'nevada': 9, 'arizona': 10, 'east': 11, 'mexican': 12, 'baja': 13, 'south': 14, 'coastline': 15, 'pacific': 16, 'ocean': 17, 'west': 18}


In [16]:
vocab_size = len(word2idx) + 1    ### Get unique words
print(vocab_size)         # california = 0+1   

19


In [17]:
sequence_data

[1, 2, 3, 4, 5, 1, 6, 7, 8, 9, 10, 11, 12, 2, 13, 1, 14, 15, 16, 17, 18]

# Matrix of Sequecne 

In [18]:
sequences = []  # empty list 
for i in range(3,len(sequence_data)): # i = 3 to 20 # sequence_data= count of words
    abc = sequence_data[i-3:i+1] # start = 0, stop = 4
    sequences.append(abc)

In [19]:
sequences

[[1, 2, 3, 4],
 [2, 3, 4, 5],
 [3, 4, 5, 1],
 [4, 5, 1, 6],
 [5, 1, 6, 7],
 [1, 6, 7, 8],
 [6, 7, 8, 9],
 [7, 8, 9, 10],
 [8, 9, 10, 11],
 [9, 10, 11, 12],
 [10, 11, 12, 2],
 [11, 12, 2, 13],
 [12, 2, 13, 1],
 [2, 13, 1, 14],
 [13, 1, 14, 15],
 [1, 14, 15, 16],
 [14, 15, 16, 17],
 [15, 16, 17, 18]]

In [20]:
import numpy as np
sequences = np.array(sequences)
sequences

array([[ 1,  2,  3,  4],
       [ 2,  3,  4,  5],
       [ 3,  4,  5,  1],
       [ 4,  5,  1,  6],
       [ 5,  1,  6,  7],
       [ 1,  6,  7,  8],
       [ 6,  7,  8,  9],
       [ 7,  8,  9, 10],
       [ 8,  9, 10, 11],
       [ 9, 10, 11, 12],
       [10, 11, 12,  2],
       [11, 12,  2, 13],
       [12,  2, 13,  1],
       [ 2, 13,  1, 14],
       [13,  1, 14, 15],
       [ 1, 14, 15, 16],
       [14, 15, 16, 17],
       [15, 16, 17, 18]])

In [21]:
X = []
Y = []
for i in sequences: 
    X.append(i[0:3])  # i = 0,1,2
    Y.append(i[3])

X = np.array(X)
Y = np.array(Y)

In [22]:
data

'california state western united states california borders oregon north nevada arizona east mexican state baja california south coastline pacific ocean west'

In [23]:
print("Data" , X[:5])
print("Response" , Y[:5])

Data [[1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 1]
 [5 1 6]]
Response [4 5 1 6 7]


In [24]:
Y

array([ 4,  5,  1,  6,  7,  8,  9, 10, 11, 12,  2, 13,  1, 14, 15, 16, 17,
       18])

In [25]:
Y=to_categorical(Y,num_classes=vocab_size)  # Matrix of Y  
Y[:5]

array([[0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.]])

# Model Building

In [26]:
model = Sequential()
model.add(Embedding(vocab_size,10, input_length=3)) # data import
model.add(LSTM(150,return_sequences=True)) # LSTM1
model.add(LSTM(150))  # LSTM2
model.add(Dense(150,activation='relu')) # HIDDEN LAYER
model.add(Dense(vocab_size, activation='softmax'))

C:\ProgramData\anaconda3\envs\dl_env\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
model.compile(optimizer='adam',loss = 'categorical_crossentropy',metrics=['accuracy'])

In [29]:
r = model.fit(X,Y,epochs=100)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 25s 25s/step - accuracy: 0.1667 - loss: 2.9442
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 930ms/step - accuracy: 0.2222 - loss: 2.9427
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - accuracy: 0.1111 - loss: 2.9410
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.1111 - loss: 2.9392
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 524ms/step - accuracy: 0.1111 - loss: 2.9372
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step - accuracy: 0.1111 - loss: 2.9350
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 515ms/step - accuracy: 0.1111 - loss: 2.9325
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.1111 - loss: 2.9297
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 641ms/step - accuracy: 0.1111 - loss: 2.9265
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 802ms/step - accuracy: 0.1111 - loss: 2.9230
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 560ms/step - accuracy: 0.1111 - loss: 2.9191
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 396ms/step - accuracy: 

# Future Word Prediction

#### Describe
* Model = model we build
* tokenizer = Breaking into Word
* enter_text = input user give

In [30]:
for word, index in tokenizer.word_index.items():  # Key = word , value = Index 
    print(word,index)

california 1
state 2
western 3
united 4
states 5
borders 6
oregon 7
north 8
nevada 9
arizona 10
east 11
mexican 12
baja 13
south 14
coastline 15
pacific 16
ocean 17
west 18


In [31]:
enter_text='states california borders'  
encoded = tokenizer.texts_to_sequences([enter_text])  # 5,1,6
encoded = np.array(encoded)
encoded
predicted= np.argmax(model.predict(encoded))  # input = 3 # prediction = 4  
predicted

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


np.int64(7)

In [32]:
def Predict_Next_Words(model,tokenizer,enter_text):
        encoded = tokenizer.texts_to_sequences([enter_text]) # in_text = 5,1,6 
        encoded = np.array(encoded) # [3]
        predicted= np.argmax(model.predict(encoded))  # input = 5,1,6 # prediction = 7  
        predicted_word=''
        for word, index in tokenizer.word_index.items():  # word = California , index = 1
            if  index==predicted:    # Predicted  7 = Index 7
                predicted_word = word      # word = OREGON
                break
        result=enter_text + ' ' + predicted_word
        return result

In [33]:
data

'california state western united states california borders oregon north nevada arizona east mexican state baja california south coastline pacific ocean west'

In [34]:
print(Predict_Next_Words(model,tokenizer,'states california borders'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
states california borders oregon


In [35]:
print(Predict_Next_Words(model,tokenizer,'california borders oregon'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
california borders oregon north


In [36]:
print(Predict_Next_Words(model,tokenizer,'coastline pacific ocean'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
coastline pacific ocean west


# Finished